# Principales determinantes del voto en Latinoamérica — Segunda entrega
Pablo Santiago Martínez Soler · Seminario M73 (BDMLIA), Maestría en Economía Aplicada, FCE-UBA.

Este notebook reproduce todos los resultados del PDF: validación temporal con ventana expansiva, hiperparámetros con validación interna, imputación dentro del pipeline, benchmark único, fuente de texto (Wikipedia) y conjunto de prueba evaluado una sola vez. Datos y scripts: https://github.com/santimsoler/Machine-learning---prediccion-electoral

## 0. Preparación
Se crean carpetas y se descargan del repositorio el panel de 102 elecciones-vuelta y las series macro anuales (Banco Mundial).

In [1]:
import os, urllib.request
for d in ['final','modelos','scripts','fig']: os.makedirs(d, exist_ok=True)
BASE='https://raw.githubusercontent.com/santimsoler/Machine-learning---prediccion-electoral/main/data/'
for f in ['panel_modelo_eleccion.csv','macro_anual_bm.csv','cuenta_corriente_wb.csv','base_final.csv']:
    urllib.request.urlretrieve(BASE+f,'final/'+f)
import pandas as pd; print(pd.read_csv('final/panel_modelo_eleccion.csv').shape)

(102, 73)


### Fuerza histórica del partido oficialista
Variable construida con los resultados previos (voto del partido del oficialismo en elecciones anteriores en que fue de los dos más votados). Se incluye la tabla ya construida.

In [2]:
fp = '''id_eleccion,fp_prev1,fp_max2,fp_media_hist,fp_nunca_top2
AR-1983-PRES-1,0.0,0.0,0.0,1
AR-1989-PRES-1,42.592,42.592,42.592,0
AR-1995-PRES-1,43.39,43.39,41.7765,0
AR-1999-PRES-1,26.353,43.39,36.635333333333335,0
AR-2003-PRES-1,19.989,26.353,32.47375,0
AR-2007-PRES-1,46.699,46.699,35.318799999999996,0
AR-2011-PRES-1,45.283,46.699,36.9795,0
AR-2015-PRES-1,54.111,54.111,39.426857142857145,0
AR-2019-PRES-1,34.153,34.153,4.269125,0
AR-2023-PRES-1,48.237,48.237,40.145111111111106,0
BO-1993-PRES-1,0.0,0.0,0.0,1
BO-1997-PRES-1,35.56,35.56,35.56,0
BO-2002-PRES-1,22.26,22.26,21.655,0
BO-2005-PRES-1,0.0,0.0,0.0,1
BO-2009-PRES-1,53.73976834164927,53.73976834164927,18.669942085412316,0
BO-2014-PRES-1,63.91154916030818,63.91154916030818,27.718263500391487,0
BO-2019-PRES-1,61.01339588952202,63.91154916030818,33.267452231913246,0
BO-2020-PRES-1,0.0,0.0,0.0,1
BO-2025-PRES-1,54.729536145175054,54.729536145175054,37.62213408057052,0
BR-2002-PRES-1,0.0,0.0,0.0,1
BR-2006-PRES-1,46.444,46.444,46.444,0
BR-2010-PRES-1,48.608,48.608,47.525999999999996,0
BR-2014-PRES-1,46.906,48.608,47.31933333333333,0
BR-2018-PRES-1,0.0,0.0,0.0,1
BR-2022-PRES-1,46.031,46.031,9.206199999999999,0
BR-2026-PRES-1,48.431,48.431,43.543499999999995,0
CL-1989-PRES-1,0.0,0.0,0.0,1
CL-1993-PRES-1,55.167,55.167,55.167,0
CL-1999-PRES-1,57.981,57.981,56.574,0
CL-2005-PRES-1,47.956,57.981,53.70133333333333,0
CL-2009-PRES-1,45.962,47.956,51.76649999999999,0
CL-2013-PRES-1,44.058,44.058,34.1592,0
CL-2017-PRES-1,46.704,46.704,47.227666666666664,0
CL-2021-PRES-1,36.643,36.643,33.21,0
CL-2025-PRES-1,25.824,25.824,41.485875,0
CO-1998-PRES-1,0.0,0.0,0.0,1
CO-2002-PRES-1,0.0,0.0,0.0,1
CO-2006-PRES-1,54.006,54.006,27.003,0
CO-2010-PRES-1,63.566,63.566,39.190666666666665,0
CO-2014-PRES-1,47.406,63.566,41.2445,0
CO-2018-PRES-1,0.0,0.0,0.0,1
CO-2022-PRES-1,0.0,0.0,0.0,1
CO-2026-PRES-1,41.051,41.051,9.511857142857142,0
CR-1990-PRES-1,0.0,0.0,0.0,1
CR-1994-PRES-1,51.5,51.5,51.5,0
CR-1998-PRES-1,49.69,49.69,48.445,0
CR-2002-PRES-1,46.96,47.74,48.73333333333334,0
CR-2006-PRES-1,38.58,46.96,46.19500000000001,0
CR-2010-PRES-1,40.92,40.92,42.684000000000005,0
CR-2014-PRES-1,46.91,46.91,43.38833333333334,0
CR-2018-PRES-1,30.64,30.64,13.64857142857143,0
CR-2022-PRES-1,21.63,30.64,14.64625,0
CR-2026-PRES-1,0.0,0.0,0.0,1
EC-1996-PRES-1,0.0,0.0,0.0,1
EC-1998-PRES-1,0.0,0.0,0.0,1
EC-2002-PRES-1,0.0,0.0,0.0,1
EC-2006-PRES-1,0.0,0.0,0.0,1
EC-2009-PRES-1,22.84,22.84,5.71,0
EC-2013-PRES-1,51.99,51.99,14.966,0
EC-2017-PRES-1,57.17,57.17,22.0,0
EC-2021-PRES-1,0.0,0.0,0.0,1
EC-2023-PRES-1,0.0,0.0,0.0,1
EC-2025-PRES-1,23.47,23.47,2.6077777777777778,0
MX-1994-PRES-1,0.0,0.0,0.0,1
MX-2000-PRES-1,50.13,50.13,50.13,0
MX-2006-PRES-1,42.52,42.52,34.605000000000004,0
MX-2012-PRES-1,35.89,42.52,35.03333333333334,0
MX-2018-PRES-1,38.21,38.21,31.112500000000004,0
MX-2024-PRES-1,53.19,53.19,10.638,0
PA-1994-PRES-1,0.0,0.0,0.0,1
PA-1999-PRES-1,33.3,33.3,33.3,0
PA-2004-PRES-1,44.8,44.8,36.945,0
PA-2009-PRES-1,47.44,47.44,39.52,0
PA-2014-PRES-1,60.03,60.03,15.0075,0
PA-2019-PRES-1,39.07,39.07,22.592000000000002,0
PA-2024-PRES-1,33.35,33.35,31.593333333333334,0
PE-2006-PRES-1,0.0,0.0,0.0,1
PE-2011-PRES-1,0.0,0.0,0.0,1
PE-2016-PRES-1,0.0,0.0,0.0,1
PE-2021-PRES-1,0.0,0.0,0.0,1
PE-2026-PRES-1,0.0,0.0,0.0,1
PY-1993-PRES-1,0.0,0.0,0.0,1
PY-1998-PRES-1,41.631404251762525,41.631404251762525,41.631404251762525,0
PY-2003-PRES-1,55.35,55.35,48.49070212588126,0
PY-2008-PRES-1,38.30703223720944,55.35,45.09614549632399,0
PY-2013-PRES-1,44.182291690421565,44.182291690421565,36.56852506965994,0
PY-2018-PRES-1,48.512073841809446,48.512073841809446,42.92367793512957,0
PY-2023-PRES-1,48.96,48.96,43.92973161260797,0
UY-2014-PRES-1,0.0,0.0,0.0,1
UY-2019-PRES-1,49.446,49.446,49.446,0
UY-2024-PRES-1,29.7,31.938,30.819,0
'''
open('final/fuerza_partido_previa.csv','w').write(fp)
print(len(fp.splitlines())-1,'filas')

91 filas


## 1. Nueva fuente de datos: texto de Wikipedia
**Cómo se obtuvo.** La celda siguiente descarga, con la API de Wikipedia en español, el artículo de cada una de las 76 elecciones (primera vuelta; la segunda hereda el texto), se queda con las secciones previas a la votación (lista de secciones permitidas y vetadas, para evitar resultados) y cuenta un diccionario de 16 temas. Se encontraron los 76 artículos (mediana: 1.385 palabras previas). Más abajo se incluye la tabla de conteos obtenida, para poder reproducir el resto sin volver a descargar.

In [3]:
# (Opcional, tarda unos minutos) Descarga y conteo desde Wikipedia. Si se activa, reemplaza la tabla incluida.
CORRER_WIKIPEDIA = False
if CORRER_WIKIPEDIA:
    import requests, re, time, collections, random, pandas as pd

    ELECCIONES = {
     'AR':[1989,1995,1999,2003,2007,2011,2015,2019,2023],
     'BO':[1993,1997,2002,2009,2014,2019,2025],
     'BR':[2002,2006,2010,2014,2018,2022,2026],
     'CL':[1989,1993,1999,2005,2009,2013,2017,2021,2025],
     'CO':[1998,2006,2010,2014,2026],
     'CR':[1990,1994,1998,2002,2006,2010,2014,2018,2022,2026],
     'EC':[2009,2013,2017,2021,2025],
     'MX':[1994,2000,2006,2012,2018,2024],
     'PA':[1994,1999,2004,2009,2014,2019,2024],
     'PE':[2021],
     'PY':[1993,1998,2003,2008,2013,2018,2023],
     'UY':[2014,2019,2024],
    }
    NOMBRE = {'AR':'Argentina','BO':'Bolivia','BR':'Brasil','CL':'Chile','CO':'Colombia','CR':'Costa Rica','EC':'Ecuador',
              'MX':'México','PA':'Panamá','PE':'Perú','PY':'Paraguay','UY':'Uruguay'}
    API = 'https://es.wikipedia.org/w/api.php'
    HDR = {'User-Agent':'TP-MEA-UBA-electoral/1.0 (trabajo academico, FCE-UBA)'}

    def api(params):
        for _ in range(4):
            try:
                r = requests.get(API, params={**params,'format':'json'}, headers=HDR, timeout=30)
                if r.status_code == 200: return r.json()
            except Exception: pass
            time.sleep(2)
        return {}

    def texto(titulo):
        j = api({'action':'query','prop':'extracts','explaintext':1,'exsectionformat':'wiki','redirects':1,'titles':titulo})
        pg = list(j.get('query',{}).get('pages',{}).values())
        if not pg or 'missing' in pg[0] or not pg[0].get('extract'): return None, None
        return pg[0]['title'], pg[0]['extract']

    def candidatos_titulo(p, a):
        P = NOMBRE[p]; anios = [str(a), f'{a}-{a+1}']
        for y in anios:
            for t in ['Elecciones presidenciales de','Elecciones generales de','Elecciones federales de','Elección presidencial de','Elecciones presidenciales y legislativas de']:
                yield f'{t} {P} de {y}'

    def buscar(p, a):
        for ti in candidatos_titulo(p, a):
            t, ext = texto(ti)
            if ext and len(ext.split()) > 80: return t, ext, 'directo'
        for q in [f'elecciones presidenciales {NOMBRE[p]} {a}', f'elección presidencial {NOMBRE[p]} {a}', f'elecciones generales {NOMBRE[p]} {a}']:
            j = api({'action':'query','list':'search','srsearch':q,'srlimit':8})
            for h in j.get('query',{}).get('search',[]):
                ti = h['title']; l = ti.lower()
                if (l.startswith('elecci') and (str(a) in ti) and NOMBRE[p].lower() in l
                    and 'parlamentaria' not in l and 'legislativa' not in l and 'municipal' not in l and 'regional' not in l):
                    t, ext = texto(ti)
                    if ext and len(ext.split()) > 80 and 'estatal' not in t.lower(): return t, ext, 'búsqueda'
        return None, None, 'no encontrado'

    ALLOW = re.compile(r'(antecedente|contexto|trasfondo|sistema electoral|reglas|legislaci|regulaci|candidat|definici[oó]n|campa[ñn]a|encuesta|sondeo|primaria|apoyo|alianza|coalici|partidos|cronograma|calendario|propuesta|debate|desarrollo|etapa|cuestiones|temas|organizaci|inscripci)', re.I)
    DENY  = re.compile(r'(resultado|escrutinio|primera vuelta|segunda vuelta|balotaje|ballottage|por (regi|prov|depart|cant)|elecciones generales|presidente y|presidenciales$|elecci[oó]n presidencial$|parlament|legislativ|consecuencia|reacciones|an[aá]lisis|tras |v[eé]ase|referencias|enlaces|notas|bibliograf|fuentes|gobernador|estatal|senador|diputad|investidura|posesi)', re.I)
    def secciones(ext):
        partes = re.split(r'\n(={2,})\s*(.+?)\s*\1\s*\n', ext); out = []; top_ok = False
        for i in range(1, len(partes)-2, 3):
            nivel, tit, cuerpo = len(partes[i]), partes[i+1].strip(), partes[i+2].strip()
            if nivel == 2: top_ok = bool(ALLOW.search(tit)) and not DENY.search(tit); ok = top_ok
            else: ok = top_ok and not DENY.search(tit)
            out.append((tit, cuerpo, ok))
        return out

    # ---- diccionario: raíces (regex) agrupadas por tema ----
    DIC = {
     'crisis':   ['crisis','recesi','inflaci','hiperinflaci','desempleo','desocupaci','pobreza','devaluaci','default','ajuste','deuda','estanflaci','corralito'],
     'corrup':   ['corrupci','esc[aá]ndalo','soborn','coima','imputad','procesad','denunci','malversaci'],
     'protesta': ['protest','huelga','paro','movilizaci','estallido','manifestaci','cacerol','bloque','descontent','malestar','disturbio','saqueo'],
     'insegur':  ['inseguridad','violenci','narco','delincuen','crimin','homicid','pandill'],
     'continu':  ['continuidad','reelecci','oficialis','sucesor','delf[ií]n','heredero'],
     'cambio':   ['cambio','alternancia','outsider','antipol','antisistema','renovaci'],
     'polariz':  ['polariz','antagonis','grieta','enfrentamiento','confrontaci'],
     'fragm':    ['fragment','divisi[oó]n','dividid','ruptura','fractura','interna','disidente','coalici'],
     'favorito': ['favorit','ventaja','encuesta','sondeo','popularidad','aprobaci','lideraba','liderazgo'],
     'bonanza':  ['crecimiento','bonanza','estabilidad','auge','prosperidad','recuperaci'],
     'izq':      ['izquierd','socialis','progresis','populis','bolivarian'],
     'der':      ['derecha','conservador','liberal','neoliberal','mercado'],
     'regimen':  ['dictadura','transici[oó]n','golpe','militar','autoritari'],
     'reforma':  ['reforma','constituci','enmienda','destituci','renuncia','inhabilit','juicio pol'],
     'pandemia': ['pandemia','covid'],
     'FUGA':     ['gan[oó]\\b','ganador','victoria','derrot','result[oó] electo','fue electo','elegido presidente','triunf'],
    }
    def contar(tp):
        t = tp.lower()
        return {tema: sum(len(re.findall(r'\b'+r, t)) for r in raices) for tema, raices in DIC.items()}

    log = []; filas = []; corpus = collections.Counter(); kw = {}
    for pais, anios in ELECCIONES.items():
        for a in anios:
            titulo, ext, how = buscar(pais, a)
            id_ = f'{pais}-{a}-PRES-1'
            if not ext:
                print('NO ENCONTRADO', pais, a); continue
            previas = [(t, c) for t, c, ok in secciones(ext) if ok]
            tp = '\n\n'.join(c for _, c in previas)
            f = dict(id_eleccion=id_, n_total=len(ext.split()), n_prev=len(tp.split()), n_secc=len(previas)); f.update(contar(tp)); filas.append(f)
            corpus.update(w for w in re.findall(r'[a-záéíóúñ]{6,}', tp.lower()))
            for tema in ['corrupci','crisis','protest']:
                m = re.search(r'.{0,110}\b'+tema+r'.{0,110}', tp.replace('\n',' '), re.I)
                if m: kw.setdefault(tema, []).append((id_, m.group(0)))
            log.append(f'{id_}|{titulo[-22:]}|{how[:3]}|prev={len(tp.split())}|nsec={len(previas)}')
            time.sleep(0.3)

    df = pd.DataFrame(filas)
    print('=====INICIO_CSV=====')
    print(df.to_csv(index=False))
    print('=====FIN_CSV=====')
    print('articulos con texto:', len(df), 'de 76 | mediana palabras previas:', df.n_prev.median())
    print('\n'.join(log))
    print('\n=== PALABRAS MÁS FRECUENTES ===')
    print(', '.join(f'{w}:{n}' for w, n in corpus.most_common(30)))
    print('\n=== CONTEXTOS PARA VALIDAR ===')
    random.seed(1)
    for tema, lst in kw.items():
        for id_, frag in random.sample(lst, min(3, len(lst))): print(f'[{tema}] {id_}: ...{frag[:200]}...')
    df.to_csv('final/texto_conteos.csv', index=False)

In [4]:
texto_csv = '''id_eleccion,n_total,n_prev,n_secc,crisis,corrup,protesta,insegur,continu,cambio,polariz,fragm,favorito,bonanza,izq,der,regimen,reforma,pandemia,FUGA
AR-1989-PRES-1,5927,4942,9,11,3,5,0,4,1,1,15,7,4,17,14,14,13,0,19
AR-1995-PRES-1,3472,2432,6,10,5,2,0,8,4,2,8,10,4,5,1,0,24,0,7
AR-1999-PRES-1,2696,2244,8,9,7,1,2,8,0,0,9,4,1,1,4,1,7,0,15
AR-2003-PRES-1,4366,1956,7,8,2,2,0,3,0,0,12,7,2,1,4,0,8,0,6
AR-2007-PRES-1,1541,1088,9,7,0,0,0,4,0,0,2,2,1,4,0,0,7,0,6
AR-2011-PRES-1,3477,2558,11,7,4,2,0,3,0,0,8,2,1,9,1,0,3,0,11
AR-2015-PRES-1,4434,591,9,0,0,0,0,2,0,0,8,1,0,2,0,0,4,0,3
AR-2019-PRES-1,11347,4422,11,15,2,5,4,13,16,1,24,14,0,23,4,3,5,0,10
AR-2023-PRES-1,6618,2464,12,6,1,1,0,10,8,0,20,4,0,8,0,0,7,1,12
BO-1993-PRES-1,656,378,4,0,0,0,0,0,1,0,1,0,0,1,1,0,0,0,0
BO-1997-PRES-1,1075,818,6,0,0,0,0,1,0,0,1,1,0,1,2,0,7,0,2
BO-2002-PRES-1,1655,1166,7,0,1,4,0,0,3,0,4,0,2,12,1,1,4,0,0
BO-2009-PRES-1,729,418,3,0,0,0,0,1,0,0,0,3,0,1,0,0,4,0,3
BO-2014-PRES-1,1550,1191,12,0,0,1,0,2,0,0,3,7,0,3,0,0,3,0,2
BO-2019-PRES-1,5670,1560,10,0,0,0,0,2,0,0,2,1,0,5,0,1,7,0,1
BO-2025-PRES-1,4994,3318,20,1,0,25,2,11,3,0,5,10,2,16,8,9,26,0,5
BR-2002-PRES-1,2265,1416,6,4,0,2,0,0,2,0,9,2,1,11,4,2,0,0,0
BR-2006-PRES-1,5574,650,2,0,4,0,0,2,1,0,3,1,0,4,0,2,2,0,7
BR-2010-PRES-1,3617,2158,12,0,1,0,0,5,0,0,5,6,0,7,3,0,5,0,0
BR-2014-PRES-1,3137,1796,6,2,1,18,0,1,1,2,5,7,1,5,2,0,3,0,3
BR-2018-PRES-1,2138,419,7,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,1
BR-2022-PRES-1,3366,1661,7,4,3,0,0,1,1,0,3,5,0,4,7,0,3,5,1
BR-2026-PRES-1,3672,2500,13,1,12,1,0,1,0,4,5,1,0,1,1,3,3,0,3
CL-1989-PRES-1,1856,1682,10,1,0,0,0,0,4,0,5,1,1,9,3,6,2,0,1
CL-1993-PRES-1,1820,1662,10,1,0,2,0,2,2,0,2,0,1,4,5,3,2,0,6
CL-1999-PRES-1,1567,361,3,2,0,0,0,1,1,0,0,1,1,1,2,1,0,0,1
CL-2005-PRES-1,5582,806,4,0,0,2,0,0,3,0,3,1,0,4,3,0,1,0,2
CL-2009-PRES-1,4883,4418,11,1,0,3,0,2,9,1,11,19,1,23,1,1,13,0,8
CL-2013-PRES-1,4195,1780,7,1,0,2,0,1,2,0,7,6,1,6,1,0,7,0,3
CL-2017-PRES-1,1611,1353,6,0,0,0,0,2,2,0,5,1,0,3,2,0,3,0,0
CL-2021-PRES-1,3673,2342,9,0,2,1,0,1,1,0,9,2,0,5,3,0,8,0,1
CL-2025-PRES-1,1519,1233,7,0,0,5,4,1,2,1,2,6,0,2,4,1,8,2,4
CO-1998-PRES-1,929,633,7,0,0,0,0,0,2,0,2,5,0,0,9,1,0,0,3
CO-2006-PRES-1,1012,836,7,0,0,0,0,2,2,0,0,2,0,1,11,1,8,0,2
CO-2010-PRES-1,2479,1989,12,1,0,0,0,4,0,0,2,1,1,0,5,4,5,0,2
CO-2014-PRES-1,1420,1042,6,1,1,0,0,4,1,0,4,0,1,3,2,0,5,0,3
CO-2026-PRES-1,8465,5826,45,0,1,1,0,4,2,0,9,4,0,3,7,0,24,0,30
CR-1990-PRES-1,855,617,7,0,0,0,1,1,0,0,3,1,0,2,3,0,1,0,1
CR-1994-PRES-1,1277,1043,4,0,1,0,3,0,1,1,0,1,0,1,7,3,0,0,2
CR-1998-PRES-1,1151,674,2,1,1,0,1,1,0,0,2,2,0,0,2,2,0,0,1
CR-2002-PRES-1,1876,1178,4,1,0,0,0,2,3,0,4,6,0,1,0,0,1,0,4
CR-2006-PRES-1,2090,1479,11,0,2,3,0,1,3,0,4,5,1,5,2,0,6,0,7
CR-2010-PRES-1,3118,669,5,0,1,0,0,0,1,0,1,0,0,1,1,1,0,0,3
CR-2014-PRES-1,6387,5071,33,1,5,4,0,5,6,1,17,10,0,6,14,1,7,0,6
CR-2018-PRES-1,10041,2909,12,2,7,1,0,2,4,2,4,5,4,4,9,0,5,0,4
CR-2022-PRES-1,6004,4242,22,5,10,5,3,1,3,0,13,13,2,8,20,1,9,8,1
CR-2026-PRES-1,2799,1717,14,1,1,1,1,4,0,1,5,1,1,2,2,0,5,0,0
EC-2009-PRES-1,1855,1118,7,0,1,1,0,2,2,0,0,5,0,1,0,3,4,0,1
EC-2013-PRES-1,3171,2271,14,5,4,1,0,6,1,0,3,3,0,6,2,0,1,0,1
EC-2017-PRES-1,6773,2040,8,0,1,0,0,10,2,0,10,18,0,6,1,0,6,0,4
EC-2021-PRES-1,5585,4096,26,1,10,2,2,3,4,0,4,0,0,5,7,4,7,11,1
EC-2025-PRES-1,8727,4327,23,1,15,2,12,2,5,1,1,0,1,1,2,5,15,0,2
MX-1994-PRES-1,2706,2367,14,0,0,2,3,4,0,0,2,9,0,1,3,2,5,0,3
MX-2000-PRES-1,1612,779,6,0,0,0,0,1,4,0,1,10,0,0,1,0,1,0,2
MX-2006-PRES-1,4496,1753,12,0,1,0,0,0,0,0,4,8,0,1,5,0,3,0,1
MX-2012-PRES-1,3657,2014,9,0,0,2,0,0,0,0,6,12,0,3,0,0,1,0,1
MX-2018-PRES-1,11980,6111,22,0,1,0,1,1,4,0,30,49,2,3,1,0,7,0,17
MX-2024-PRES-1,4040,2289,11,3,3,4,13,2,1,0,2,4,1,0,0,1,9,2,7
PA-1994-PRES-1,434,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
PA-1999-PRES-1,94,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
PA-2004-PRES-1,241,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
PA-2009-PRES-1,757,384,1,0,0,0,0,0,3,0,0,0,0,0,1,0,0,0,1
PA-2014-PRES-1,397,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
PA-2019-PRES-1,567,387,7,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,2
PA-2024-PRES-1,3332,1526,4,2,2,2,0,0,2,0,4,0,0,0,1,0,1,2,0
PE-2021-PRES-1,9481,3591,14,5,13,6,2,1,4,2,5,5,3,7,10,3,15,15,11
PY-1993-PRES-1,868,3,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
PY-1998-PRES-1,1682,263,2,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,4
PY-2003-PRES-1,180,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
PY-2008-PRES-1,664,248,4,0,0,0,0,2,2,0,2,0,0,2,1,0,1,0,1
PY-2013-PRES-1,787,366,5,0,0,0,0,2,0,0,2,1,0,1,1,0,2,0,1
PY-2018-PRES-1,704,278,5,0,0,2,0,1,0,0,0,1,0,0,0,3,5,0,0
PY-2023-PRES-1,1310,824,7,0,5,0,2,1,1,0,2,1,0,2,2,0,1,0,1
UY-2014-PRES-1,1795,365,3,0,0,0,0,0,1,0,3,0,0,1,0,0,0,0,2
UY-2019-PRES-1,2658,116,3,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0
UY-2024-PRES-1,16031,12300,15,9,12,5,3,18,4,0,58,7,2,7,2,15,19,5,20
'''
if not os.path.exists('final/texto_conteos.csv'):
    open('final/texto_conteos.csv','w').write(texto_csv)
t = pd.read_csv('final/texto_conteos.csv'); print(t.shape, '| artículos con >=150 palabras previas:', int((t.n_prev>=150).sum()))
t.head()

(76, 20) | artículos con >=150 palabras previas: 69


,id_eleccion,n_total,n_prev,n_secc,crisis,corrup,protesta,insegur,continu,cambio,polariz,fragm,favorito,bonanza,izq,der,regimen,reforma,pandemia,FUGA
0,AR-1989-PRES-1,5927,4942,9,11,3,5,0,4,1,1,15,7,4,17,14,14,13,0,19
1,AR-1995-PRES-1,3472,2432,6,10,5,2,0,8,4,2,8,10,4,5,1,0,24,0,7
2,AR-1999-PRES-1,2696,2244,8,9,7,1,2,8,0,0,9,4,1,1,4,1,7,0,15
3,AR-2003-PRES-1,4366,1956,7,8,2,2,0,3,0,0,12,7,2,1,4,0,8,0,6
4,AR-2007-PRES-1,1541,1088,9,7,0,0,0,4,0,0,2,2,1,4,0,0,7,0,6


**Transformación y unión.** Conteos por cada 1.000 palabras; cuatro índices fijados de antemano (malestar, continuidad−cambio, fragmentación, izquierda−derecha). La unión con el panel es por `id_eleccion` (la segunda vuelta usa el texto de su primera). Las 7 elecciones con menos de 150 palabras previas quedan sin dato y se imputan dentro del pipeline.

## 2. Núcleo de validación
`e2_core.py` define: carga y variables; bloques de validación por año con ventana expansiva (entrena solo con elecciones estrictamente anteriores); validación interna temporal para elegir hiperparámetros; pipelines con imputación por mediana dentro de cada partición; benchmark (promedio histórico por vuelta).

In [5]:
%%writefile scripts/e2_core.py
"""Segunda entrega - núcleo de validación.
Panel crudo (con NaN) -> imputación DENTRO del pipeline; ventana expansiva por año; hiperparámetros con validación interna temporal.
"""
import numpy as np, pandas as pd, warnings
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
warnings.filterwarnings('ignore')

CORTE_TEST = pd.Timestamp('2022-01-01')   # elecciones desde esta fecha = conjunto de prueba (se toca una sola vez)
PRIMER_ANIO_VAL = 2010                     # primer año de la validación en ventana expansiva

BASE = ['oficialismo_dudoso','oficialismo_va_por_reeleccion','aprobacion_oficial_completa',
        'bm_inflacion_ipc_tm1','us_current_account_tm1','vuelta','fp_prev1','fp_nunca_top2']
TEXTO = ['malestar','cont_cambio','fragm_txt','izq_der']
AMPLIO = BASE + ['ideologia_of','percep_econ_lb','bm_crecimiento_pib_tm1','bm_desempleo_oit_tm1','us_fiscal_primario_tm1',
        'us_ner_var_tm1','us_cuenta_corriente_wb_tm1','of_tradicional','anio_c','pobnac_685_tm1','pobnac_365_tm1',
        'd_pobnac_685_mandato','d_pobnac_365_mandato','anios_mandato','of_top2_eleccion_previa','of_n_top2_previas']

def slog(x): return np.sign(x)*np.log1p(np.abs(x))

def cargar():
    e = pd.read_csv('final/panel_modelo_eleccion.csv'); fp = pd.read_csv('final/fuerza_partido_previa.csv')
    key = e.id_eleccion.str[:-1]+'1'                      # la 2a vuelta hereda la fuerza previa de su 1a vuelta
    for c in ['fp_prev1','fp_nunca_top2']: e[c] = key.map(dict(zip(fp.id_eleccion, fp[c]))).fillna(0).values
    e['fecha'] = pd.to_datetime(e.fecha)
    for c in ['bm_inflacion_ipc_tm1','us_ner_var_tm1']: e[c] = slog(e[c])
    # resultado de la 1a vuelta (conocido antes de la 2a): NaN en las primeras vueltas, se imputa dentro del pipeline
    r1 = e[e.vuelta==1].set_index('id_eleccion').y_share_nacional
    e['r1_share'] = np.where(e.vuelta==2, key.map(r1), np.nan)
    t = pd.read_csv('final/texto_conteos.csv'); ok = t.n_prev >= 150; k = 1000/t.n_prev.where(ok)
    t['malestar'] = (t.crisis+t.corrup+t.protesta+t.insegur)*k; t['cont_cambio'] = (t.continu-t.cambio)*k
    t['fragm_txt'] = t.fragm*k; t['izq_der'] = (t.izq-t.der)*k; t['leak_txt'] = t.FUGA*k
    for c in ['malestar','cont_cambio','fragm_txt','izq_der','leak_txt']: e[c] = key.map(dict(zip(t.id_eleccion, t[c])))   # la 2a vuelta hereda el texto de su 1a
    e = e.sort_values(['fecha','id_eleccion']).reset_index(drop=True)
    e['colapso'] = (e.y_share_nacional < 10).astype(int)
    return e

def bloques(fechas, idx, desde_anio):
    """Bloques de validación = un año calendario; entrena solo con elecciones estrictamente anteriores al bloque."""
    f = fechas.iloc[idx]; out = []
    for a in sorted(f.dt.year.unique()):
        if a < desde_anio: continue
        va = idx[(f.dt.year == a).values]; ini = fechas.iloc[va].min()
        tr = idx[(f < ini).values]
        if len(tr) >= 15: out.append((tr, va, a))
    return out

def cv_interno(fechas, tr, k=5):
    """Validación interna dentro de la partición: ventana expansiva sobre los últimos k años del entrenamiento."""
    res = bloques(fechas, tr, int(fechas.iloc[tr].dt.year.min()))
    res = res[-k:]
    return [(np.where(np.isin(tr, a))[0], np.where(np.isin(tr, b))[0]) for a, b, _ in res]

def pipe(modelo, cols):
    imp = SimpleImputer(strategy='median')
    if modelo in ('Ridge','ElasticNet'):
        est = Ridge() if modelo=='Ridge' else ElasticNet(max_iter=20000)
        return Pipeline([('imp',imp),('sc',StandardScaler()),('m',est)])
    est = (RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=1) if modelo=='RandomForest'
           else GradientBoostingRegressor(subsample=0.7, random_state=0))
    return Pipeline([('imp',imp),('m',est)])

GRIDS = {
 'Ridge': {'m__alpha': list(np.logspace(-2,3,12))},
 'ElasticNet': {'m__alpha': [0.01,0.05,0.2,0.5,1,3], 'm__l1_ratio':[0.3,0.7]},
 'RandomForest': {'m__max_depth':[2,3,5], 'm__min_samples_leaf':[3,6], 'm__max_features':[0.5,1.0]},
 'GradBoost': {'m__learning_rate':[0.02,0.05], 'm__max_depth':[1,2,3], 'm__n_estimators':[100,300]},
}

def ajustar_predecir(df, modelo, cols, tr, te, fechas):
    """Ajusta con hiperparámetros elegidos por validación interna (solo con tr) y predice te."""
    X = df[cols]; y = df.y_share_nacional.values
    if modelo == 'Benchmark':   # promedio histórico por vuelta, solo con entrenamiento
        m = {v: y[tr][df.vuelta.values[tr]==v].mean() for v in (1,2)}
        return np.array([m[v] for v in df.vuelta.values[te]]), {}
    gs = GridSearchCV(pipe(modelo, cols), GRIDS[modelo], cv=cv_interno(fechas, tr), scoring='neg_mean_absolute_error', n_jobs=2)
    gs.fit(X.iloc[tr], y[tr])
    return gs.predict(X.iloc[te]), gs.best_params_

def validacion_externa(df, modelo, cols, idx_train):
    """Ventana expansiva sobre el entrenamiento. Devuelve predicciones por elección y MAE por bloque."""
    fechas = df.fecha; filas = []; hp = []
    for tr, va, a in bloques(fechas, idx_train, PRIMER_ANIO_VAL):
        p, b = ajustar_predecir(df, modelo, cols, tr, va, fechas); hp.append(b)
        for i, pi in zip(va, p): filas.append((df.id_eleccion[i], a, df.y_share_nacional[i], pi))
    R = pd.DataFrame(filas, columns=['id_eleccion','bloque','y','pred']); R['err'] = (R.y-R.pred).abs()
    mae_b = R.groupby('bloque').err.mean()
    return R, dict(MAE_pool=R.err.mean(), RMSE=np.sqrt(((R.y-R.pred)**2).mean()),
                   R2=1-((R.y-R.pred)**2).sum()/((R.y-R.y.mean())**2).sum(),
                   MAE_fold_media=mae_b.mean(), MAE_fold_desvio=mae_b.std(), n=len(R), folds=len(mae_b)), hp


Writing scripts/e2_core.py


## 3. Etapa 1 — Validación en entrenamiento (elecciones anteriores a 2022)
Compara el benchmark con Ridge, ElasticNet, Random Forest y Gradient Boosting, en tres conjuntos de variables, con todas las elecciones y sin colapsos. Tarda varios minutos.

In [6]:
"""Etapa 1: validación en ventana expansiva SOLO con elecciones de entrenamiento (< 2022). No toca el conjunto de prueba."""
import sys; sys.path.insert(0,'scripts')
import pandas as pd, numpy as np, time
import e2_core as c
df = c.cargar()
res = []; preds = []
SETS = {'Base':c.BASE, 'Base+R1':c.BASE+['r1_share'], 'Amplio+R1':c.AMPLIO+['r1_share']}
MODELOS = {'Base':['Ridge','ElasticNet','RandomForest','GradBoost'], 'Base+R1':['Ridge','ElasticNet','RandomForest','GradBoost'],
           'Amplio+R1':['Ridge','RandomForest','GradBoost']}
for muestra, filtro in [('todas', df.colapso>=0), ('sin_colapsos', df.colapso==0)]:
    d = df[filtro].reset_index(drop=True)
    tr = np.where(d.fecha < c.CORTE_TEST)[0]
    t0=time.time()
    R, m, _ = c.validacion_externa(d, 'Benchmark', c.BASE, tr)
    res.append(dict(muestra=muestra, set='-', modelo='Benchmark', **m)); R.assign(muestra=muestra, set='-', modelo='Benchmark').pipe(preds.append)
    for s, cols in SETS.items():
        for mod in MODELOS[s]:
            R, m, hp = c.validacion_externa(d, mod, cols, tr)
            res.append(dict(muestra=muestra, set=s, modelo=mod, **m)); preds.append(R.assign(muestra=muestra, set=s, modelo=mod))
            print(muestra, s, mod, {k:round(v,2) for k,v in m.items()}, f'{time.time()-t0:.0f}s', flush=True)
            pd.DataFrame(res).to_csv('modelos/e2_cv_resultados.csv', index=False)
            pd.concat(preds).to_csv('modelos/e2_cv_predicciones.csv', index=False)
print('FIN')


todas Base Ridge {'MAE_pool': np.float64(7.73), 'RMSE': np.float64(9.96), 'R2': np.float64(0.66), 'MAE_fold_media': np.float64(6.58), 'MAE_fold_desvio': 3.59, 'n': 41, 'folds': 10} 9s
todas Base ElasticNet {'MAE_pool': np.float64(7.51), 'RMSE': np.float64(9.81), 'R2': np.float64(0.67), 'MAE_fold_media': np.float64(6.43), 'MAE_fold_desvio': 3.59, 'n': 41, 'folds': 10} 14s
todas Base RandomForest {'MAE_pool': np.float64(9.87), 'RMSE': np.float64(12.25), 'R2': np.float64(0.49), 'MAE_fold_media': np.float64(9.41), 'MAE_fold_desvio': 3.45, 'n': 41, 'folds': 10} 266s
todas Base GradBoost {'MAE_pool': np.float64(9.11), 'RMSE': np.float64(11.07), 'R2': np.float64(0.58), 'MAE_fold_media': np.float64(9.23), 'MAE_fold_desvio': 2.21, 'n': 41, 'folds': 10} 356s
todas Base+R1 Ridge {'MAE_pool': np.float64(7.71), 'RMSE': np.float64(10.14), 'R2': np.float64(0.65), 'MAE_fold_media': np.float64(6.58), 'MAE_fold_desvio': 3.72, 'n': 41, 'folds': 10} 361s
todas Base+R1 ElasticNet {'MAE_pool': np.float64(7.

## 4. Etapa 2 — Aporte de cada variable y concentración del error
Mismo modelo (Ridge) con y sin cada variable, sobre las mismas particiones, con IC por bootstrap sobre elecciones. También lista las elecciones donde se concentra el error.

In [7]:
"""Etapa 2 (solo entrenamiento): aporte de cada variable = mismo modelo con y sin ella, mismas particiones; concentración del error."""
import sys; sys.path.insert(0,'scripts')
import pandas as pd, numpy as np, e2_core as c
df=c.cargar(); out=[]; P={}
rng=np.random.default_rng(0)
for muestra,f in [('todas',df.colapso>=0),('sin_colapsos',df.colapso==0)]:
    d=df[f].reset_index(drop=True); tr=np.where(d.fecha<c.CORTE_TEST)[0]
    R0,m0,_=c.validacion_externa(d,'Ridge',c.BASE,tr); P[(muestra,'completo')]=R0
    out.append(dict(muestra=muestra,variable='(modelo completo)',MAE=m0['MAE_pool'],dMAE=0,lo=0,hi=0))
    for v in c.BASE:
        cols=[x for x in c.BASE if x!=v]
        R,m,_=c.validacion_externa(d,'Ridge',cols,tr)
        dif=(R.set_index('id_eleccion').err-R0.set_index('id_eleccion').err).values   # >0: sin la variable empeora
        bs=[rng.choice(dif,len(dif)).mean() for _ in range(3000)]
        out.append(dict(muestra=muestra,variable=v,MAE=m['MAE_pool'],dMAE=dif.mean(),lo=np.percentile(bs,2.5),hi=np.percentile(bs,97.5)))
        print(muestra,v,round(m['MAE_pool'],2),round(dif.mean(),2),flush=True)
pd.DataFrame(out).to_csv('modelos/e2_ablacion.csv',index=False)
# concentración del error
for k,R in P.items():
    R=R.sort_values('err',ascending=False); tot=R.err.sum()
    print('\n',k,'MAE',round(R.err.mean(),2),'| top5 aportan',round(R.err.head(5).sum()/tot*100),'% del error absoluto | top10',round(R.err.head(10).sum()/tot*100),'%')
    print(R.head(8).round(1)[['id_eleccion','y','pred','err']].to_string(index=False))
    R.to_csv(f'modelos/e2_errores_{k[0]}.csv',index=False)
# horizonte 90 dias: share de aprobaciones medidas a <=90 dias de la eleccion
e=pd.read_csv('final/panel_modelo_eleccion.csv'); g=e[e.vuelta==1].aprob_ead_gap_dias
print('\nAprobación EAD con gap<=90d:',(g<=90).sum(),'de',g.notna().sum(), '| otras fuentes:', e[e.vuelta==1].aprobacion_fuente_completa.ne('EAD').sum())


todas oficialismo_dudoso 8.66 0.93
todas oficialismo_va_por_reeleccion 7.69 -0.04
todas aprobacion_oficial_completa 9.5 1.77
todas bm_inflacion_ipc_tm1 7.92 0.19
todas us_current_account_tm1 8.08 0.35
todas vuelta 9.26 1.53
todas fp_prev1 8.05 0.32
todas fp_nunca_top2 8.02 0.29
sin_colapsos oficialismo_dudoso 8.36 0.37
sin_colapsos oficialismo_va_por_reeleccion 7.87 -0.12
sin_colapsos aprobacion_oficial_completa 9.8 1.8
sin_colapsos bm_inflacion_ipc_tm1 8.15 0.15
sin_colapsos us_current_account_tm1 8.48 0.49
sin_colapsos vuelta 9.26 1.26
sin_colapsos fp_prev1 8.33 0.34
sin_colapsos fp_nunca_top2 8.26 0.27

 ('todas', 'completo') MAE 7.73 | top5 aportan 33 % del error absoluto | top10 53 %
   id_eleccion    y  pred  err
CR-2018-PRES-2 60.6  31.9 28.7
CO-2014-PRES-1 25.7  49.6 23.9
PA-2019-PRES-1 10.8  29.4 18.5
PA-2014-PRES-1 31.4  49.1 17.7
CO-2010-PRES-2 69.1  52.3 16.8
PY-2018-PRES-1 49.0  33.7 15.2
BR-2018-PRES-1  1.2  14.7 13.5
CL-2013-PRES-1 24.9  37.3 12.5

 ('sin_colapsos', 'com

## 5. Etapa 3 — ¿Aportan las variables de texto?
Base contra Base + índices de texto, mismo modelo y mismas particiones.

In [8]:
"""Etapa 3 (solo entrenamiento): ¿aporta la fuente de texto? Mismo modelo con/sin, mismas particiones."""
import sys; sys.path.insert(0,'scripts')
import pandas as pd, numpy as np, e2_core as c
df=c.cargar(); rng=np.random.default_rng(0); out=[]
V={'Base':c.BASE,'Base+Texto(4 índices)':c.BASE+c.TEXTO,'Base+cont_cambio':c.BASE+['cont_cambio'],'Base+malestar':c.BASE+['malestar'],'Base+fragm_txt':c.BASE+['fragm_txt'],'Base+izq_der':c.BASE+['izq_der']}
for muestra,f in [('todas',df.colapso>=0),('sin_colapsos',df.colapso==0)]:
    d=df[f].reset_index(drop=True); tr=np.where(d.fecha<c.CORTE_TEST)[0]
    for mod in ['Ridge','ElasticNet']:
        R0=None
        for nom,cols in V.items():
            R,m,_=c.validacion_externa(d,mod,cols,tr)
            if nom=='Base': R0=R
            dif=(R.set_index('id_eleccion').err-R0.set_index('id_eleccion').err).values  # <0: mejora con texto
            bs=[rng.choice(dif,len(dif)).mean() for _ in range(3000)]
            out.append(dict(muestra=muestra,modelo=mod,set=nom,MAE=m['MAE_pool'],MAE_fold=m['MAE_fold_media'],SD_fold=m['MAE_fold_desvio'],R2=m['R2'],d_vs_base=dif.mean(),lo=np.percentile(bs,2.5),hi=np.percentile(bs,97.5)))
            print(muestra,mod,nom,round(m['MAE_pool'],2),round(dif.mean(),2),flush=True)
pd.DataFrame(out).to_csv('modelos/e2_texto.csv',index=False)


todas Ridge Base 7.73 0.0
todas Ridge Base+Texto(4 índices) 7.95 0.22
todas Ridge Base+cont_cambio 8.02 0.29
todas Ridge Base+malestar 7.73 0.0
todas Ridge Base+fragm_txt 7.48 -0.24
todas Ridge Base+izq_der 7.83 0.1
todas ElasticNet Base 7.51 0.0
todas ElasticNet Base+Texto(4 índices) 8.18 0.67
todas ElasticNet Base+cont_cambio 7.95 0.44
todas ElasticNet Base+malestar 7.88 0.38
todas ElasticNet Base+fragm_txt 7.59 0.09
todas ElasticNet Base+izq_der 7.78 0.27
sin_colapsos Ridge Base 8.0 0.0
sin_colapsos Ridge Base+Texto(4 índices) 8.02 0.02
sin_colapsos Ridge Base+cont_cambio 8.43 0.44
sin_colapsos Ridge Base+malestar 8.18 0.19
sin_colapsos Ridge Base+fragm_txt 8.01 0.01
sin_colapsos Ridge Base+izq_der 8.19 0.2
sin_colapsos ElasticNet Base 8.11 0.0
sin_colapsos ElasticNet Base+Texto(4 índices) 8.07 -0.04
sin_colapsos ElasticNet Base+cont_cambio 8.39 0.28
sin_colapsos ElasticNet Base+malestar 8.24 0.13
sin_colapsos ElasticNet Base+fragm_txt 8.05 -0.06
sin_colapsos ElasticNet Base+izq_der

## 6. Etapa 4 — Conjunto de prueba (2022–2026), una sola vez
Modelo elegido por regla fijada antes de mirar la prueba: Ridge con el set Base (menor MAE promedio entre muestras; ante diferencias menores a 0,1, el más simple). Se compara con el benchmark. **No volver a ejecutar para ajustar el modelo.**

In [9]:
"""Etapa 4: conjunto de prueba (elecciones >= 2022), UNA sola vez. Modelo elegido en etapa 1 por regla fija: Ridge, set Base."""
import sys; sys.path.insert(0,'scripts')
import pandas as pd, numpy as np, e2_core as c, matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
df=c.cargar(); rng=np.random.default_rng(0); res=[]; allp=[]; coefs={}
for muestra,f in [('todas',df.colapso>=0),('sin_colapsos',df.colapso==0)]:
    d=df[f].reset_index(drop=True); tr=np.where(d.fecha<c.CORTE_TEST)[0]; te=np.where(d.fecha>=c.CORTE_TEST)[0]
    y=d.y_share_nacional.values[te]
    for mod in ['Benchmark','Ridge']:
        p,hp=c.ajustar_predecir(d,mod,c.BASE,tr,te,d.fecha)
        e=np.abs(y-p); bs=[rng.choice(e,len(e)).mean() for _ in range(3000)]
        res.append(dict(muestra=muestra,modelo=mod,n=len(te),MAE=e.mean(),lo=np.percentile(bs,2.5),hi=np.percentile(bs,97.5),RMSE=np.sqrt(((y-p)**2).mean()),R2=1-((y-p)**2).sum()/((y-y.mean())**2).sum(),hp=str(hp)))
        allp.append(pd.DataFrame(dict(muestra=muestra,modelo=mod,id_eleccion=d.id_eleccion.values[te],y=y,pred=p,err=e)))
    # coeficientes del modelo final ajustado con todo el entrenamiento
    gs=c.GridSearchCV(c.pipe('Ridge',c.BASE),c.GRIDS['Ridge'],cv=c.cv_interno(d.fecha,tr),scoring='neg_mean_absolute_error').fit(d[c.BASE].iloc[tr],d.y_share_nacional.values[tr])
    coefs[muestra]=pd.Series(gs.best_estimator_[-1].coef_,index=c.BASE)
R=pd.DataFrame(res); P=pd.concat(allp); R.to_csv('modelos/e2_test_resultados.csv',index=False); P.to_csv('modelos/e2_test_predicciones.csv',index=False)
pd.DataFrame(coefs).to_csv('modelos/e2_coeficientes.csv')
print(R.round(2).to_string(index=False)); print(P[P.modelo=='Ridge'].sort_values('err',ascending=False).round(1).head(10).to_string(index=False))


     muestra    modelo  n   MAE   lo    hi  RMSE    R2                                            hp
       todas Benchmark 20 10.96 6.77 15.87 15.07  0.18                                            {}
       todas     Ridge 20  7.73 4.82 11.21 10.73  0.58   {'m__alpha': np.float64(1.873817422860383)}
sin_colapsos Benchmark 17  6.26 4.07  9.01  8.16  0.06                                            {}
sin_colapsos     Ridge 17  7.50 4.36 11.22 10.53 -0.57 {'m__alpha': np.float64(0.08111308307896872)}
     muestra modelo    id_eleccion    y  pred  err
       todas  Ridge CR-2026-PRES-1 48.3  16.5 31.8
sin_colapsos  Ridge CR-2026-PRES-1 48.3  18.3 30.0
       todas  Ridge MX-2024-PRES-1 61.2  40.8 20.3
sin_colapsos  Ridge MX-2024-PRES-1 61.2  41.2 20.0
       todas  Ridge PY-2023-PRES-1 42.7  25.6 17.2
sin_colapsos  Ridge PY-2023-PRES-1 42.7  28.0 14.8
       todas  Ridge CR-2022-PRES-1  0.7  11.7 11.1
       todas  Ridge BR-2026-PRES-1 45.2  34.9 10.3
       todas  Ridge CO-2026-PRES-1 4

## 7. Robustez: sin los oficialismos dudosos, y exploración solo con datos de entrenamiento
Repite el modelo final excluyendo las 8 elecciones con oficialismo dudoso (y combinando con la exclusión de colapsos). También calcula las correlaciones **solo con elecciones anteriores a 2022**, informando el n de cada una.

In [10]:
"""Etapa 5: (a) modelo final sin los 8 oficialismos dudosos; (b) exploración SOLO con datos de entrenamiento (< 2022)."""
import sys; sys.path.insert(0,'scripts')
import pandas as pd, numpy as np, e2_core as c
df=c.cargar(); rng=np.random.default_rng(0); out=[]
cols=[x for x in c.BASE if x!='oficialismo_dudoso']      # sin la marca (constante cuando se excluyen los dudosos)
for nombre,f in [('todas',df.colapso>=0),('sin dudosos',df.oficialismo_dudoso==0),('sin colapsos',df.colapso==0),('sin dudosos ni colapsos',(df.oficialismo_dudoso==0)&(df.colapso==0))]:
    d=df[f].reset_index(drop=True); tr=np.where(d.fecha<c.CORTE_TEST)[0]; te=np.where(d.fecha>=c.CORTE_TEST)[0]; y=d.y_share_nacional.values[te]
    fila=dict(muestra=nombre,n_total=len(d),n_val=None,n_test=len(te))
    for mod in ['Benchmark','Ridge']:
        R,m,_=c.validacion_externa(d,mod,cols if mod=='Ridge' else c.BASE,tr)
        fila[f'val_MAE_{mod}']=m['MAE_pool']; fila['n_val']=m['n']; fila[f'val_foldSD_{mod}']=m['MAE_fold_desvio']
        p,_=c.ajustar_predecir(d,mod,cols if mod=='Ridge' else c.BASE,tr,te,d.fecha); fila[f'test_MAE_{mod}']=np.abs(y-p).mean()
    out.append(fila); print({k:(round(v,2) if isinstance(v,float) else v) for k,v in fila.items()},flush=True)
pd.DataFrame(out).to_csv('modelos/e2_dudosos.csv',index=False)
# exploración solo con entrenamiento
tr=df[df.fecha<c.CORTE_TEST]; V=['d_pobnac_685_mandato','d_pobnac_365_mandato','aprobacion_oficial_completa','bm_inflacion_ipc_tm1','us_current_account_tm1','fp_prev1','oficialismo_va_por_reeleccion','oficialismo_dudoso','vuelta']
E_=[]
for v in V:
    z=tr[[v,'y_share_nacional']].dropna(); z1=tr[tr.vuelta==1][[v,'y_share_nacional']].dropna()
    E_.append(dict(variable=v,n_todas=len(z),r_pearson=z.corr().iloc[0,1],r_spearman=z.corr(method='spearman').iloc[0,1],n_R1=len(z1),r_R1=z1.corr().iloc[0,1]))
E_=pd.DataFrame(E_); E_.to_csv('modelos/e2_eda_entrenamiento.csv',index=False); print(E_.round(2).to_string(index=False))
print(tr.groupby('vuelta').y_share_nacional.agg(['count','mean','std']).round(1))


{'muestra': 'todas', 'n_total': 102, 'n_val': 41, 'n_test': 20, 'val_MAE_Benchmark': np.float64(12.05), 'val_foldSD_Benchmark': 8.26, 'test_MAE_Benchmark': np.float64(10.96), 'val_MAE_Ridge': np.float64(8.66), 'val_foldSD_Ridge': 3.98, 'test_MAE_Ridge': np.float64(8.2)}
{'muestra': 'sin dudosos', 'n_total': 94, 'n_val': 38, 'n_test': 18, 'val_MAE_Benchmark': np.float64(10.4), 'val_foldSD_Benchmark': 6.63, 'test_MAE_Benchmark': np.float64(9.64), 'val_MAE_Ridge': np.float64(7.87), 'val_foldSD_Ridge': 3.75, 'test_MAE_Ridge': np.float64(6.76)}
{'muestra': 'sin colapsos', 'n_total': 93, 'n_val': 38, 'n_test': 17, 'val_MAE_Benchmark': np.float64(10.05), 'val_foldSD_Benchmark': 6.62, 'test_MAE_Benchmark': np.float64(6.26), 'val_MAE_Ridge': np.float64(8.36), 'val_foldSD_Ridge': 3.99, 'test_MAE_Ridge': np.float64(6.92)}
{'muestra': 'sin dudosos ni colapsos', 'n_total': 90, 'n_val': 37, 'n_test': 16, 'val_MAE_Benchmark': np.float64(9.62), 'val_foldSD_Benchmark': 6.53, 'test_MAE_Benchmark': np.fl

## 8. Segundas vueltas por separado (con la 1ª vuelta como predictor)
Modelos aparte para segundas vueltas (media histórica, regresión sobre la 1ª vuelta, Ridge con 1ª vuelta) y para primeras vueltas, frente al modelo conjunto, con la misma validación. Se adopta un modelo aparte solo si mejora al menos 1 punto de MAE; la prueba no se usa para decidir.

In [11]:
"""Etapa 6: modelo aparte para segundas vueltas (con la 1a vuelta como predictor) y modelo aparte de primeras vueltas, frente al modelo conjunto.
Validación: ventana expansiva por año (2010-2021) sobre elecciones de entrenamiento; la prueba (>=2022) NO se usa para elegir."""
import sys; sys.path.insert(0,'scripts')
import pandas as pd, numpy as np, e2_core as c
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
df=c.cargar(); rng=np.random.default_rng(0)
R2COLS=['r1_share','aprobacion_oficial_completa','fp_prev1','oficialismo_va_por_reeleccion']
def mod_r2(kind):
    if kind=='OLS_r1': return Pipeline([('imp',SimpleImputer(strategy='median')),('m',LinearRegression())]), ['r1_share']
    if kind=='Ridge_r1+3': return Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),('m',RidgeCV(alphas=np.logspace(-1,3,15)))]), R2COLS
def cv_r2(d, kind, desde=2010, hasta=2021, minimo=6):
    rows=[]; f=d.fecha
    for a in range(desde,hasta+1):
        va=np.where((f.dt.year==a)&(d.vuelta==2))[0]
        if len(va)==0: continue
        ini=f.iloc[np.where(f.dt.year==a)[0]].min(); tr=np.where((f<ini)&(d.vuelta==2))[0]
        if len(tr)<minimo: continue
        y=d.y_share_nacional.values
        if kind=='Benchmark_R2': p=np.repeat(y[tr].mean(),len(va))
        elif kind=='y2=y1': p=d.r1_share.values[va]
        else:
            m,cols=mod_r2(kind); m.fit(d[cols].iloc[tr],y[tr]); p=m.predict(d[cols].iloc[va])
        rows+= [(d.id_eleccion[i],a,y[i],pi) for i,pi in zip(va,p)]
    R=pd.DataFrame(rows,columns=['id_eleccion','bloque','y','pred']); R['err']=(R.y-R.pred).abs(); return R
res=[]
for muestra,filt in [('todas',df.colapso>=0),('sin_colapsos',df.colapso==0)]:
    d=df[filt].reset_index(drop=True); tr=np.where(d.fecha<c.CORTE_TEST)[0]
    # modelos conjuntos: predicciones de la etapa 1 (misma validación) restringidas a R2
    for nom,cols in [('Conjunto Ridge Base',c.BASE),('Conjunto Ridge Base+R1',c.BASE+['r1_share'])]:
        R,m,_=c.validacion_externa(d,'Ridge',cols,tr); R['vuelta']=R.id_eleccion.str[-1].astype(int)
        z=R[R.vuelta==2]; res.append(dict(muestra=muestra,vuelta='R2',modelo=nom,n=len(z),MAE=z.err.mean(),RMSE=np.sqrt(((z.y-z.pred)**2).mean())))
        z=R[R.vuelta==1]; res.append(dict(muestra=muestra,vuelta='R1',modelo=nom,n=len(z),MAE=z.err.mean(),RMSE=np.sqrt(((z.y-z.pred)**2).mean())))
        if nom=='Conjunto Ridge Base': Rj=R.copy()
    for kind in ['Benchmark_R2','y2=y1','OLS_r1','Ridge_r1+3']:
        R=cv_r2(d,kind); res.append(dict(muestra=muestra,vuelta='R2',modelo='Aparte '+kind,n=len(R),MAE=R.err.mean(),RMSE=np.sqrt(((R.y-R.pred)**2).mean())))
        if kind=='OLS_r1': Ro=R.copy()
    # modelo aparte de 1a vuelta: Ridge solo con R1 (sin 'vuelta' ni r1)
    d1=d[d.vuelta==1].reset_index(drop=True); t1=np.where(d1.fecha<c.CORTE_TEST)[0]
    cols1=[x for x in c.BASE if x!='vuelta']
    R,m,_=c.validacion_externa(d1,'Ridge',cols1,t1); res.append(dict(muestra=muestra,vuelta='R1',modelo='Aparte Ridge solo R1',n=len(R),MAE=R.err.mean(),RMSE=np.sqrt(((R.y-R.pred)**2).mean())))
    # diferencia pareada R2: aparte OLS_r1 vs conjunto Base (mismas elecciones)
    a=Ro.set_index('id_eleccion').err; b=Rj.set_index('id_eleccion').err; k=a.index.intersection(b.index); dif=(a[k]-b[k]).values
    bs=[rng.choice(dif,len(dif)).mean() for _ in range(3000)]
    print(muestra,'R2 aparte OLS_r1 - conjunto Base: ',round(dif.mean(),2),np.percentile(bs,[2.5,97.5]).round(2),'n=',len(k))
R=pd.DataFrame(res); R.to_csv('modelos/e2_segunda_vuelta.csv',index=False); print(R.round(2).to_string(index=False))
# coeficientes del modelo R2 simple ajustado con todas las R2 de entrenamiento
d=df.copy(); t=d[(d.vuelta==2)&(d.fecha<c.CORTE_TEST)]; z=t[['r1_share','y_share_nacional']].dropna(); lr=LinearRegression().fit(z[['r1_share']],z.y_share_nacional)
print('R2 train n=',len(z),'y2 = %.1f + %.2f*y1'%(lr.intercept_,lr.coef_[0]),' corr',round(z.corr().iloc[0,1],2))


todas R2 aparte OLS_r1 - conjunto Base:  -0.31 [-3.38  3.86] n= 13
sin_colapsos R2 aparte OLS_r1 - conjunto Base:  -1.34 [-3.72  1.71] n= 13
     muestra vuelta                 modelo  n   MAE  RMSE
       todas     R2    Conjunto Ridge Base 13  7.60 10.57
       todas     R1    Conjunto Ridge Base 28  7.79  9.66
       todas     R2 Conjunto Ridge Base+R1 13  7.68 11.09
       todas     R1 Conjunto Ridge Base+R1 28  7.72  9.66
       todas     R2    Aparte Benchmark_R2 13  7.41 11.26
       todas     R2           Aparte y2=y1 13 15.91 18.26
       todas     R2          Aparte OLS_r1 13  7.30 10.39
       todas     R2      Aparte Ridge_r1+3 13  7.16 10.16
       todas     R1   Aparte Ridge solo R1 28  7.85  9.62
sin_colapsos     R2    Conjunto Ridge Base 13  8.64 11.25
sin_colapsos     R1    Conjunto Ridge Base 25  7.66  9.98
sin_colapsos     R2 Conjunto Ridge Base+R1 13  8.71 11.68
sin_colapsos     R1 Conjunto Ridge Base+R1 25  7.66  9.97
sin_colapsos     R2    Aparte Benchmark_R2 13  

## 9. Manifesto Project: cobertura y validación de la ideología
Subí `MPDataset_MPDS2026a.csv` (Manifesto Project) a la carpeta de trabajo para correr esta celda. Cuenta cuántos programas hay por elección y compara el RILE con la ideología codificada a mano.

In [12]:
"""Etapa 7: cobertura del Manifesto Project (MPDS2026a) sobre las 76 elecciones y validación débil de la ideología codificada a mano.
Requiere subir MPDataset_MPDS2026a.csv a la carpeta de trabajo."""
import pandas as pd, numpy as np, re, os
ruta='MPDataset_MPDS2026a.csv'
if not os.path.exists(ruta):
    print('Para correr esta etapa, subí MPDataset_MPDS2026a.csv (Manifesto Project) a esta carpeta.')
else:
    m=pd.read_csv(ruta,low_memory=False)
    cc={'Argentina':'AR','Bolivia':'BO','Brazil':'BR','Chile':'CL','Colombia':'CO','Costa Rica':'CR','Ecuador':'EC','Mexico':'MX','Panama':'PA','Peru':'PE','Paraguay':'PY','Uruguay':'UY'}
    m=m[m.countryname.isin(cc)].copy(); m['pais']=m.countryname.map(cc); m['anio']=(m.date//100).astype(int)
    e=pd.read_csv('final/panel_modelo_eleccion.csv'); e1=e[e.vuelta==1]
    filas=[]; hit=[]
    for r in e1.itertuples():
        z=m[(m.pais==r.pais)&(m.anio==r.anio)]; filas.append(dict(id_eleccion=r.id_eleccion,pais=r.pais,anio=r.anio,n_programas=len(z)))
        f=str(r.familia_of).lower().strip()
        k=z[(z.partyabbrev.fillna('').str.lower()==f)|(z.partyname.fillna('').str.lower().str.contains(re.escape(f)) if len(f)>3 else False)]
        if len(k): hit.append((r.id_eleccion,k.rile.iloc[0],r.ideologia_of))
    C=pd.DataFrame(filas); C['cubierta']=C.n_programas>0
    print('Elecciones:',len(C),'| con >=1 programa:',int(C.cubierta.sum()),'| >=3:',int((C.n_programas>=3).sum()))
    print('antes de 2000:',int(C[C.anio<2000].cubierta.sum()),'de',(C.anio<2000).sum(),'| 2000 en adelante:',int(C[C.anio>=2000].cubierta.sum()),'de',(C.anio>=2000).sum())
    print(C.groupby('pais').agg(elecciones=('id_eleccion','count'),cubiertas=('cubierta','sum'),programas=('n_programas','sum')).T)
    H=pd.DataFrame(hit,columns=['id','rile','ideologia_of']).dropna()
    print('Programa del oficialismo identificable por nombre en',len(hit),'elecciones; corr(RILE, ideología codificada): pearson %.2f, spearman %.2f (n=%d)'%(H.corr(numeric_only=True).iloc[0,1],H.corr(method='spearman',numeric_only=True).iloc[0,1],len(H)))


Elecciones: 76 | con >=1 programa: 51 | >=3: 45
antes de 2000: 13 de 17 | 2000 en adelante: 38 de 59
pais        AR  BO  BR  CL  CO  CR  EC  MX  PA  PE  PY  UY
elecciones   9   7   7   9   5  10   5   6   7   1   7   3
cubiertas    8   2   6   8   4   8   3   5   6   0   0   1
programas   29   9  20  30  19  30  12  30  26   0   0   5
Programa del oficialismo identificable por nombre en 15 elecciones; corr(RILE, ideología codificada): pearson 0.40, spearman 0.16 (n=13)


## 10. Robustez económica: sudden stop y efectos por país
Primero se arma el panel macro anual y una **proxy** de sudden stop (no hay flujos de capital: mejora de la cuenta corriente de al menos 3 puntos del PBI con contracción del producto; umbrales fijados antes de ver resultados). La variable se mide en la **segunda mitad del mandato del oficialismo** (los últimos ceil(m/2) años antes del año electoral, con m = duración del mandato), no en una ventana fija de dos años; además se prueba la caída anual de reservas de al menos 15 % (7 países, panel de la tesis). Luego se prueban, solo con datos de entrenamiento: una marca de Argentina con interacciones, pendientes económicas por país, e interacciones estimadas con IC por bootstrap.

In [13]:
"""Panel macro anual por país (12 países) y proxy de sudden stop; se guarda en final/macro_panel_ss.csv.
Usa macro_anual_bm.csv (Banco Mundial) y cuenta_corriente_wb.csv (descargados del repositorio). No hay datos de flujos de capital: es una proxy de ajuste externo forzado."""
import pandas as pd
a=pd.read_csv('final/macro_anual_bm.csv'); cc=pd.read_csv('final/cuenta_corriente_wb.csv')
m={'ARG':'AR','BOL':'BO','BRA':'BR','CHL':'CL','COL':'CO','CRI':'CR','ECU':'EC','MEX':'MX','PAN':'PA','PER':'PE','PRY':'PY','URY':'UY'}
a['pais']=a.pais.map(m)
w=a.pivot_table(index=['pais','anio'],columns='variable',values='valor').reset_index().merge(cc.rename(columns={'cuenta_corriente_wb':'ca'}),on=['pais','anio'],how='left').sort_values(['pais','anio'])
w['dca']=w.groupby('pais').ca.diff(); w['dg']=w.groupby('pais').crecimiento_pib.diff(); w['depre']=w.groupby('pais').tc_oficial_lcu_usd.pct_change()*100
w['ss']=((w.dca>=3)&((w.crecimiento_pib<=0)|(w.dg<=-3))).astype(int)   # umbrales fijados antes de ver resultados
w.to_csv('final/macro_panel_ss.csv',index=False); print(int(w.ss.sum()),'años-país con proxy de',len(w))


24 años-país con proxy de 462


In [14]:
"""Etapa 8 (solo entrenamiento): efectos económicos distintos en Argentina / por país. (El sudden stop en la 2ª mitad del mandato está en la etapa 8b.)"""
import sys; sys.path.insert(0,'scripts')
import pandas as pd, numpy as np, e2_core as c
df=c.cargar(); rng=np.random.default_rng(0)
w=pd.read_csv('final/macro_panel_ss.csv').set_index(['pais','anio'])
df['AR']=(df.pais=='AR').astype(int)
df['AR_infl']=df.AR*df.bm_inflacion_ipc_tm1; df['AR_fisc']=df.AR*df.us_current_account_tm1; df['AR_aprob']=df.AR*df.aprobacion_oficial_completa
GR=[p for p,n in df[df.vuelta==1].pais.value_counts().items() if n>=6]   # países con >=6 elecciones
for p in GR:
    df[f'{p}_infl']=(df.pais==p)*df.bm_inflacion_ipc_tm1; df[f'{p}_fisc']=(df.pais==p)*df.us_current_account_tm1
B=c.BASE
S={'Base':B,
   'Base + Argentina (dummy)':B+['AR'],'Base + AR e interacciones con inflación y saldo fiscal':B+['AR','AR_infl','AR_fisc'],
   'Base + AR e interacciones con inflación, saldo fiscal y aprobación':B+['AR','AR_infl','AR_fisc','AR_aprob'],
   'Base + pendientes económicas por país (>=6 elecciones)':B+[f'{p}_{k}' for p in GR for k in ('infl','fisc')]}
out=[]; print('países con >=6 elecciones:',GR)
for muestra,f in [('todas',df.colapso>=0),('sin_colapsos',df.colapso==0)]:
    d=df[f].reset_index(drop=True); tr=np.where(d.fecha<c.CORTE_TEST)[0]; R0=None
    for n,cols in S.items():
        R,m,_=c.validacion_externa(d,'Ridge',cols,tr)
        if R0 is None: R0=R
        dif=(R.set_index('id_eleccion').err-R0.set_index('id_eleccion').err).values; bs=[rng.choice(dif,len(dif)).mean() for _ in range(2000)]
        out.append(dict(muestra=muestra,modelo=n,MAE=m['MAE_pool'],R2=m['R2'],d_vs_base=dif.mean(),lo=np.percentile(bs,2.5),hi=np.percentile(bs,97.5)))
        print(f'{muestra:13s} {n:68s} MAE {m["MAE_pool"]:.2f} Δ {dif.mean():+.2f} [{np.percentile(bs,2.5):+.2f},{np.percentile(bs,97.5):+.2f}]',flush=True)
pd.DataFrame(out).to_csv('modelos/e2_ss_hetero.csv',index=False)
# descriptivo: sudden stop vs voto (entrenamiento, primeras vueltas) y por país
tr=df[(df.fecha<c.CORTE_TEST)&(df.vuelta==1)]
print('\nCorrelación de Spearman dentro de cada país (entrenamiento, 1ª vuelta, n>=6):')
rows=[]
for p in GR:
    z=tr[tr.pais==p]; r={'pais':p,'n':len(z)}
    for v in ['bm_inflacion_ipc_tm1','us_current_account_tm1','bm_crecimiento_pib_tm1','aprobacion_oficial_completa']:
        q=z[[v,'y_share_nacional']].dropna(); r[v]=q.corr(method='spearman').iloc[0,1] if len(q)>=5 else np.nan
    rows.append(r)
T=pd.DataFrame(rows); T.to_csv('modelos/e2_corr_por_pais.csv',index=False); print(T.round(2).to_string(index=False))
# interacción Argentina: OLS con variables estandarizadas, IC por bootstrap de elecciones (entrenamiento)
d=df[df.fecha<c.CORTE_TEST].copy(); cols=B+['AR','AR_infl','AR_fisc']; X=d[cols].copy()
X=X.fillna(X.median()); mu,sd=X.mean(),X.std().replace(0,1); Z=((X-mu)/sd).values; y=d.y_share_nacional.values
def ols(Z,y):
    A=np.c_[np.ones(len(Z)),Z]; return np.linalg.lstsq(A,y,rcond=None)[0][1:]
b0=ols(Z,y); bs=np.array([ols(Z[i],y[i]) for i in (rng.integers(0,len(y),len(y)) for _ in range(2000))])
T2=pd.DataFrame({'coef':b0,'lo':np.percentile(bs,2.5,axis=0),'hi':np.percentile(bs,97.5,axis=0)},index=cols); T2.to_csv('modelos/e2_interaccion_AR.csv'); print('\nOLS estandarizado con interacciones de Argentina (n=%d):'%len(y)); print(T2.round(2).to_string())


países con >=6 elecciones: ['CR', 'AR', 'CL', 'PY', 'BO', 'PA', 'BR', 'MX']
todas         Base                                                                 MAE 7.73 Δ +0.00 [+0.00,+0.00]
todas         Base + Argentina (dummy)                                             MAE 7.58 Δ -0.15 [-0.72,+0.43]
todas         Base + AR e interacciones con inflación y saldo fiscal               MAE 8.48 Δ +0.75 [-0.33,+1.85]
todas         Base + AR e interacciones con inflación, saldo fiscal y aprobación   MAE 9.29 Δ +1.56 [+0.32,+2.89]
todas         Base + pendientes económicas por país (>=6 elecciones)               MAE 9.44 Δ +1.71 [-0.12,+3.72]
sin_colapsos  Base                                                                 MAE 8.00 Δ +0.00 [+0.00,+0.00]
sin_colapsos  Base + Argentina (dummy)                                             MAE 7.96 Δ -0.03 [-0.58,+0.52]
sin_colapsos  Base + AR e interacciones con inflación y saldo fiscal               MAE 8.97 Δ +0.97 [-0.17,+2.13]
sin_colapsos

### 10b. Sudden stop en la segunda mitad del mandato
Ventana [t − ceil(m/2), t − 1]; no se usa el año de la elección. Se compara el Ridge base con cada variable agregada, con los mismos bloques de validación.

In [15]:
"""Etapa 8b (solo entrenamiento): sudden stop en la SEGUNDA MITAD DEL MANDATO del oficialismo (no 'dos años antes').
Ventana = [t-ceil(m/2), t-1], con t = año electoral y m = duración del mandato (anios_mandato). Datos anuales del año t no se usan (ex ante).
Indicadores: (1) proxy WB (dCA>=3pp PIB y crecimiento<=0 o caída>=3pp), (2) depreciación máxima, (3) caída de reservas>=15% anual (base_final, 7 países)."""
import sys, math; sys.path.insert(0,'scripts')
import pandas as pd, numpy as np, e2_core as c
df=c.cargar(); rng=np.random.default_rng(0)
m=pd.read_csv('final/panel_modelo_eleccion.csv').set_index('id_eleccion').anios_mandato
df['m']=df.id_eleccion.map(m).fillna(4).astype(int)
w=pd.read_csv('final/macro_panel_ss.csv').set_index(['pais','anio'])
b=pd.read_csv('final/base_final.csv')
iso={'ARG':'AR','BRA':'BR','CHL':'CL','COL':'CO','PRY':'PY','PER':'PE','URY':'UY'}
b=b[b.iso3c.isin(iso)].assign(pais=lambda x:x.iso3c.map(iso)).set_index(['pais','year'])
b['perd_res']=(b.d_ln_res<=math.log(0.85)).astype(float).where(b.d_ln_res.notna())
def vent(r,tab,col,f=max):
    ys=range(r.anio-math.ceil(r.m/2),r.anio); v=[tab[col].get((r.pais,y),np.nan) for y in ys]; v=[x for x in v if pd.notna(x)]
    return f(v) if v else np.nan
df['ss_m']=[vent(r,w,'ss') for r in df.itertuples()]
df['dep_m']=[np.log1p(max(vent(r,w,'depre'),0)/100) if pd.notna(vent(r,w,'depre')) else np.nan for r in df.itertuples()]
df['res_m']=[vent(r,b,'perd_res') for r in df.itertuples()]
df['n_ss_m']=[sum(w['ss'].get((r.pais,y),0) or 0 for y in range(r.anio-math.ceil(r.m/2),r.anio)) for r in df.itertuples()]
df[['id_eleccion','pais','anio','m','ss_m','dep_m','res_m','n_ss_m']].to_csv('modelos/e2_ss_mandato_vars.csv',index=False)
tr=df[df.fecha<c.CORTE_TEST]
print('Elecciones con sudden stop (proxy) en 2ª mitad del mandato: total',int((df.ss_m==1).sum()),'/',int(df.ss_m.notna().sum()),'| entrenamiento',int((tr.ss_m==1).sum()))
print('Con caída de reservas >=15%: total',int((df.res_m==1).sum()),'/',int(df.res_m.notna().sum()),'| entrenamiento',int((tr.res_m==1).sum()))
print(df[df.ss_m==1][['id_eleccion','m','y_share_nacional']].to_string(index=False))
t1=tr[tr.vuelta==1]; print('\nVoto medio oficialismo (1ª vuelta, entrenamiento) según proxy en 2ª mitad del mandato:'); print(t1.groupby('ss_m').y_share_nacional.agg(['count','mean','median']).round(1))
print('según caída de reservas:'); print(t1.groupby('res_m').y_share_nacional.agg(['count','mean','median']).round(1))
B=c.BASE
S={'Base':B,'Base + sudden stop 2ª mitad del mandato':B+['ss_m'],'Base + depreciación máx. 2ª mitad':B+['dep_m'],'Base + caída de reservas 2ª mitad (7 países)':B+['res_m'],'Base + las tres':B+['ss_m','dep_m','res_m']}
out=[]
for muestra,f in [('todas',df.colapso>=0),('sin_colapsos',df.colapso==0)]:
    d=df[f].reset_index(drop=True); trn=np.where(d.fecha<c.CORTE_TEST)[0]; R0=None
    for n,cols in S.items():
        R,mm,_=c.validacion_externa(d,'Ridge',cols,trn)
        if R0 is None: R0=R
        dif=(R.set_index('id_eleccion').err-R0.set_index('id_eleccion').err).values; bs=[rng.choice(dif,len(dif)).mean() for _ in range(2000)]
        out.append(dict(muestra=muestra,modelo=n,MAE=mm['MAE_pool'],R2=mm['R2'],d_vs_base=dif.mean(),lo=np.percentile(bs,2.5),hi=np.percentile(bs,97.5)))
        print(f'{muestra:13s} {n:48s} MAE {mm["MAE_pool"]:.2f} Δ {dif.mean():+.2f} [{np.percentile(bs,2.5):+.2f},{np.percentile(bs,97.5):+.2f}]',flush=True)
pd.DataFrame(out).to_csv('modelos/e2_ss_mandato.csv',index=False)


Elecciones con sudden stop (proxy) en 2ª mitad del mandato: total 9 / 100 | entrenamiento 7
Con caída de reservas >=15%: total 6 / 53 | entrenamiento 6
   id_eleccion  m  y_share_nacional
AR-2003-PRES-1  4         22.245595
PY-2003-PRES-1  5         38.300000
PA-2004-PRES-1  5         16.380000
CR-2010-PRES-1  4         46.910000
EC-2017-PRES-1  4         39.360000
EC-2017-PRES-2  4         51.160000
CL-2021-PRES-1  4         12.730636
EC-2025-PRES-1  2         44.170000
EC-2025-PRES-2  2         55.630000

Voto medio oficialismo (1ª vuelta, entrenamiento) según proxy en 2ª mitad del mandato:
      count  mean  median
ss_m                     
0.0      55  35.9    38.6
1.0       6  29.3    30.3
según caída de reservas:
       count  mean  median
res_m                     
0.0       24  35.5    39.7
1.0        5  40.4    38.3
todas         Base                                             MAE 7.73 Δ +0.00 [+0.00,+0.00]
todas         Base + sudden stop 2ª mitad del mandato          MAE 8.

### Variables económicas relativas al promedio histórico del país
Crecimiento, inflación y saldo externo del año previo menos el promedio del propio país hasta dos años antes (sin mirar el futuro).

In [16]:
"""Etapa 9 (solo entrenamiento): variables económicas relativas al promedio histórico del propio país (sin mirar el futuro)."""
import sys; sys.path.insert(0,'scripts')
import pandas as pd, numpy as np, e2_core as c
df=c.cargar(); w=pd.read_csv('final/macro_panel_ss.csv'); w['linf']=np.sign(w.inflacion_ipc)*np.log1p(w.inflacion_ipc.abs()); rng=np.random.default_rng(0)
def rel(r,col):
    z=w[(w.pais==r.pais)]; v=z[z.anio==r.anio-1][col]; h=z[z.anio<=r.anio-2][col].dropna()
    return (v.iloc[0]-h.mean()) if len(v) and pd.notna(v.iloc[0]) and len(h)>=3 else np.nan
for k,col in [('g_rel','crecimiento_pib'),('infl_rel','linf'),('ca_rel','ca')]: df[k]=[rel(r,col) for r in df.itertuples()]
B=c.BASE; econ=['bm_inflacion_ipc_tm1','us_current_account_tm1']
S={'Base':B,'Base + 3 variables relativas al país':B+['g_rel','infl_rel','ca_rel'],'Base + crecimiento relativo':B+['g_rel'],'Base sin economía en niveles + relativas':[x for x in B if x not in econ]+['g_rel','infl_rel','ca_rel']}
out=[]
for muestra,f in [('todas',df.colapso>=0),('sin_colapsos',df.colapso==0)]:
    d=df[f].reset_index(drop=True); tr=np.where(d.fecha<c.CORTE_TEST)[0]; R0=None
    for n,cols in S.items():
        R,m,_=c.validacion_externa(d,'Ridge',cols,tr)
        if R0 is None: R0=R
        dif=(R.set_index('id_eleccion').err-R0.set_index('id_eleccion').err).values; bs=[rng.choice(dif,len(dif)).mean() for _ in range(2000)]
        out.append(dict(muestra=muestra,modelo=n,MAE=m['MAE_pool'],d_vs_base=dif.mean(),lo=np.percentile(bs,2.5),hi=np.percentile(bs,97.5)))
        print(f'{muestra:13s} {n[:60]:60s} MAE {m["MAE_pool"]:.2f} Δ {dif.mean():+.2f} [{np.percentile(bs,2.5):+.2f},{np.percentile(bs,97.5):+.2f}]',flush=True)
pd.DataFrame(out).to_csv('modelos/e2_relativas.csv',index=False)


todas         Base                                                         MAE 7.73 Δ +0.00 [+0.00,+0.00]
todas         Base + 3 variables relativas al país                         MAE 8.03 Δ +0.30 [-0.05,+0.68]
todas         Base + crecimiento relativo                                  MAE 8.17 Δ +0.44 [+0.16,+0.75]
todas         Base sin economía en niveles + relativas                     MAE 8.01 Δ +0.28 [-0.92,+1.51]
sin_colapsos  Base                                                         MAE 8.00 Δ +0.00 [+0.00,+0.00]
sin_colapsos  Base + 3 variables relativas al país                         MAE 8.37 Δ +0.37 [-0.10,+0.86]
sin_colapsos  Base + crecimiento relativo                                  MAE 8.21 Δ +0.21 [-0.05,+0.52]
sin_colapsos  Base sin economía en niveles + relativas                     MAE 8.32 Δ +0.33 [-0.79,+1.49]


## 11. Figuras
Predicho contra observado y residuos en la prueba.

In [17]:
import matplotlib.pyplot as plt, numpy as np
P=pd.read_csv('modelos/e2_test_predicciones.csv'); q=P[P.muestra=='todas']
fig,ax=plt.subplots(1,2,figsize=(9,3.3))
for mod,col in [('Benchmark','#999'),('Ridge','#2a6')]:
    z=q[q.modelo==mod]; ax[0].scatter(z.y,z.pred,c=col,s=22,label=mod)
ax[0].plot([0,70],[0,70],'k--',lw=.8); ax[0].set_xlabel('Observado (%)'); ax[0].set_ylabel('Predicho (%)'); ax[0].legend()
z=q[q.modelo=='Ridge']; ax[1].hist(z.y-z.pred,bins=10,color='#2a6'); ax[1].set_xlabel('Residuo (observado − predicho)')
plt.tight_layout(); plt.show()

## 12. Lectura de resultados
Ver el PDF (secciones 7 a 11): el Ridge con 8 variables mejora al benchmark en validación; en prueba mejora con todas las elecciones pero no sin colapsos (diferencias no significativas con 20 y 17 elecciones); boosting y random forest empeoran; la fuente de texto no mejora el error. Limitaciones: pocas observaciones, dependencia entre elecciones, texto escrito después del desenlace, horizonte de 90 días incompleto.